In [1]:
import pandas as pd
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_columns', 50)
import warnings
warnings.filterwarnings('ignore')

path1 = "https://raw.githubusercontent.com/Soyoung-Yoon/bigdata/main/"


**9회-1번) 대출액 차이 분석**

은행과 자본 대출 데이터를 활용하여 각 지역에 대해 연도 및 성별별 총대출액을 집계하고, 지역 간 대출 격차를 분석하려 한다.

다음 절차에 따라 문제를 해결하시오.
- 1) 각 지역(Region_Code)에 대해 연도(Year) 및 성별(Gender)별로 총대출액의 합계를 구하시오.
> 총대출액은 은행대출(Bank_Loan) + 캐피탈대출(Capital_Loan)으로 계산한다.
- 2) 총대출액의 연도 및 성별별 합계액 차이의 절댓값이 가장 큰 지역을 구해, 지역(Region_Code)을 정수형으로 출력합니다.

In [ ]:
df = pd.read_csv(path1 + "loan_data.csv")
# print(df.head(3))

# 1. 각 지역(Region_Code)에 대해 연도(Year) 및 성별(Gender)별로 총대출액의 합계를 구하시오.
# 총대출액은 은행대출(Bank_Loan) + 캐피탈대출(Capital_Loan)으로 계산한다.
df['All_Loan'] = df['Bank_Loan'] + df['Capital_Loan']
df_year = df.groupby(['Region_Code','Year'], as_index=False)['All_Loan'].sum()
df_gender = df.groupby(['Region_Code','Gender'], as_index=False)['All_Loan'].sum()
# print(df_year, df_gender, sep="\n")

# 2. 총대출액의 연도 및 성별별 합계액 차이의 절댓값이 가장 큰 지역을 구해, 지역(Region_Code)을 정수형으로 출력합니다.
df2 = df_year.merge(df_gender, "left", "Region_Code")
df2['All_Loan_diff'] = abs(df2['All_Loan_x'] - df2['All_Loan_y'])
# print(df2.head())
print(int(df2[df2['All_Loan_diff'] == df2['All_Loan_diff'].max()]['Region_Code'].values[0])) # 4146510700


4146510700


**9회-2번) 연도별 최고 검거율 범죄 유형 분석**

범죄 발생 및 검거 데이터를 기반으로 범죄유형별 검거율을 계산하고, 각 연도별로 검거율이 가장 높은 범죄유형의 검거 실적을 분석한다.

다음 절차에 따라 문제를 해결하시오.
- 1) 검거율은 다음과 같이 계산한다.
> 검거율 = 검거건수 / 발생건수
- 2) 범죄유형별로 연도별 검거율을 계산하시오.
- 3) 각 연도별로 검거율이 가장 높은 범죄유형을 찾으시오.
- 4) 해당 범죄유형의 검거건수를 구하고, 그 값들을 모두 합한 값을 정수형으로 출력하시오.

In [ ]:
df = pd.read_csv(path1 + "crime_data.csv")
# print(df.head(3))

# 1. 검거율은 다음과 같이 계산한다.
# 검거율 = 검거건수 / 발생건수

# 2. 범죄유형별로 연도별 검거율을 계산하시오.
cols = [col for col in df.columns if '유형' in col]
df = pd.melt(df, id_vars=['연도','구분'], value_vars=cols, var_name='유형', value_name='건수')
df = pd.pivot_table(df, index=['유형','연도'], columns='구분', values='건수').reset_index()
df['검거율'] = df['검거건수'] / df['발생건수']
# print(df.head(3))

# 3. 각 연도별로 검거율이 가장 높은 범죄유형을 찾으시오.
crim_type = df[df.index.isin(df.groupby('연도')['검거율'].idxmax())]['유형'].values

# 4. 해당 범죄유형의 검거건수를 구하고, 그 값들을 모두 합한 값을 정수형으로 출력하시오.
print(int(df[df['유형'].isin(crim_type)]['검거건수'].sum())) # 39055

39055


**9회-3번) 근속연수 및 교육참가 분석**

사원 데이터의 결측값을 조건에 따라 적절히 처리하고, 부서와 조건별 평균을 활용하여 특정 계산을 수행한다.

다음 절차에 따라 문제를 해결하시오.
- 1) 평균만족도에 결측치가 있는 경우, 전체 평균만족도의 평균값으로 채우시오.
- 2) 근속연수에 결측치가 있는 경우,
같은 부서와 같은 성과등급을 가진 사원들의 근속연수 평균을 정수로 변환하여 채우시오.
- 3) 변수 A는 부서가 'Sales'이고 성과등급이 'C'인 사원들의 평균 근속연수로 정의하시오.
- 4) 변수 B는 부서가 'Operations'이고 평균만족도가 2.5 이상인 사원들의 평균 교육참가횟수로 정의하시오.
- 5) 최종적으로 A + B의 값을 정수로 출력하시오.

In [ ]:
df = pd.read_csv(path1 + "hr_data.csv")
# print(df.head(3))
# print(df.isna().sum().to_frame().T)

# 1. 평균만족도에 결측치가 있는 경우, 전체 평균만족도의 평균값으로 채우시오.
df['평균만족도'] = df['평균만족도'].fillna(df['평균만족도'].mean())
# print(df.isna().sum().to_frame().T)

# 2. 근속연수에 결측치가 있는 경우, 같은 부서와 같은 성과등급을 가진 사원들의 근속연수 평균을 정수로 변환하여 채우시오.
work_mean = round(df.groupby(['부서','성과등급'])['근속연수'].transform('mean'))
df['근속연수'] = df['근속연수'].fillna(work_mean)
# print(df.isna().sum().to_frame().T)

# 3. 변수 A는 부서가 'Sales'이고 성과등급이 'C'인 사원들의 평균 근속연수로 정의하시오.
A = df[(df['부서']=='Sales') & (df['성과등급']=='C')]['근속연수'].mean()

# 4. 변수 B는 부서가 'Operations'이고 평균만족도가 2.5 이상인 사원들의 평균 교육참가횟수로 정의하시오.
B = df[(df['부서']=='Operations') & (df['평균만족도']>=2.5)]['교육참가횟수'].mean()

# 5. 최종적으로 A + B의 값을 정수로 출력하시오.
print(int(A + B)) # 20

11.0
20


**17-4) 연도별 최고 고용비중 산업 분석**

산업별 고용 데이터를 활용하여 각 연도별 고용비중이 가장 높은 산업을 분석한다.

다음 절차에 따라 문제를 해결하시오.
- 1) 산업(Industry)별 연도(Year)별 고용 인원을 기반으로, 전년도 대비 고용 증가율을 다음과 같이 계산하시오.
> 고용 증가율 = (이번년도 고용 - 전년도 고용) / 전년도 고용
- 2) 각 연도별로 고용 증가율이 가장 높은 산업을 찾으시오.
- 3) 2)에서 찾은 산업의 고용 증가량을 구하시오.
> 고용 증가량 = (이번년도 고용 - 전년도 고용)
- 4) 연도별로 구한, 고용 증가량을 합산하여 정수형으로 출력하시오.
- 단, 전년도 데이터가 존재하는 연도(2011년 이상)에 대해서만 계산한다.


In [54]:
df = pd.read_csv(path1 + "employment_data.csv")
# print(df.head(3))
# print(df['Year'].unique())

# 1. 산업(Industry)별 연도(Year)별 고용 인원을 기반으로, 전년도 대비 고용 증가율을 다음과 같이 계산하시오.
# 고용 증가율 = (이번년도 고용 - 전년도 고용) / 전년도 고용
df = df.set_index('Year')
df['Prev_Employment'] = df['Employment'].shift(1)
df['Employment_diff'] = df['Employment'].diff(1)
df = df[df.index != 2010] # 단, 전년도 데이터가 존재하는 연도(2011년 이상)만 추출

# 2. 각 연도별로 고용 증가율이 가장 높은 산업을 찾으시오.
df = df.reset_index()
df.groupby(['Year','Industry'])['Employment_diff'].max()

Year  Industry      
2011  Agriculture      -465322.0
      Construction     -264545.0
      Education          86832.0
      Energy            722381.0
      Finance            16132.0
      Healthcare         87705.0
      IT               -433932.0
      Manufacturing     384093.0
      Retail           -237077.0
      Transportation    288275.0
2012  Agriculture       162670.0
      Construction      -21620.0
      Education         592702.0
      Energy            -29397.0
      Finance          -577956.0
      Healthcare        449928.0
      IT               -391246.0
      Manufacturing     396436.0
      Retail           -688388.0
      Transportation    132441.0
2013  Agriculture       121108.0
      Construction      505308.0
      Education        -343456.0
      Energy           -127776.0
      Finance            13954.0
      Healthcare        227437.0
      IT               -163114.0
      Manufacturing      74460.0
      Retail            227951.0
      Transportation  